In [1]:
from pathlib import Path


import pandas as pd
import numpy as np

from five_safes_tes_workbench.workbench import Workbench
from partialstats.partials import SumOfSquaresPartial
from partialstats.combiners.statistical import count_combiner, mean_combiner, variance_combiner

## Full example for Mean & Variance Analysis using the 5s-TES workbench

[The 5s-TES workbench](https://github.com/federated-research/5S-TES-Workbench) provides a set of tools for interacting with Five Safes TES.
You provide the workbench your credentials for connecting to Five Safes TES, and it will configure your connection to the submission layer, including writing TES messages to your specification and collecting results.
The full details of how to use the workbench can be found in its README.
Here we will not focus on those details, but on how to use it to carry out an analysis.

For this example, the configuration is held in a `config.yml` file, as described in [the workbench](https://github.com/federated-research/5S-TES-Workbench/blob/main/example-config.yml).

For this demonstration, if you have access to the University of Nottingham submission layer, the project configuration is:

- project: "DelphiDemo"
- tes_base_url: "https://api.5s-tes.federated-research.com"
- minio_sts_endpoint: "https://api.minio.5s-tes.federated-research.com/sts"
- minio_endpoint: "https://api.minio.5s-tes.federated-research.com"
- minio_output_bucket: "126104output"
- tres:
    - "Nottingham TRE 01"
    - "Nottingham TRE 02"

For authentication, you will need to either get an access token from the submission layer user interface, or ask your administrator for keycloak details.

In [2]:
wb = Workbench()

wb.validate(config_path="config.yml") #type: ignore

INFO | Template registered: 'hello_world'
INFO | Template registered: 'custom'
INFO | Template registered: 'simple_sql'
INFO | Template registered: 'bunny'
INFO | Validation successful
INFO | Config: project='DelphiDemo' tes_base_url='https://api.5s-tes.federated-research.com/' minio_sts_endpoint='https://api.minio.5s-tes.federated-research.com/sts' minio_endpoint='https://api.minio.5s-tes.federated-research.com/' minio_output_bucket='126104output' tres=['Nottingham TRE 01', 'Nottingham TRE 02']
INFO | Auth mode: AuthMode.CREDENTIALS


## Define SQL query

The query compare systolic blood pressure between two groups:

- people with primary malignant neoplasm of skin
- people without primary malignant neoplasm of skin

The two OMOP concept IDs used are:

- `3004249`: systolic blood pressure
- `139750`: primary malignant neoplasm of skin

In [3]:
sys_pressure_neoplasm_query = """
WITH last_occurrence AS (
    SELECT
        person_id,
        value_as_number,
        ROW_NUMBER() OVER (
            PARTITION BY person_id
            ORDER BY measurement_datetime DESC NULLS LAST
        ) AS rn
    FROM "DelphiDemo".measurement
    WHERE measurement_concept_id = 3004249
      AND value_as_number IS NOT NULL
),

value_with_status AS (
    SELECT
        value_as_number,
        CASE
            WHEN person_id IN (
                SELECT person_id
                FROM "DelphiDemo".condition_occurrence
                WHERE condition_concept_id = 139750
            )
            THEN 'with'
            ELSE 'without'
        END AS condition_status
    FROM last_occurrence
    WHERE rn = 1
)

SELECT
    condition_status,
    COUNT(value_as_number) AS count,
    SUM(value_as_number) AS sum,
    SUM(value_as_number * value_as_number) AS sumsq
FROM value_with_status
GROUP BY condition_status;
"""

wb.build_tes.simple_sql(
    name="Mean and variance systolic blood pressure",
    query=sys_pressure_neoplasm_query
)

wb.submit()

INFO | Building TES task from template: 'simple_sql'
INFO | Resolving template: 'simple_sql'
INFO | TES Task built successfully
INFO | TES payload:
{
   "name": "Mean and variance systolic blood pressure",
   "description": "Simple SQL Task",
   "outputs": [
      {
         "url": "s3://",
         "path": "/outputs",
         "type": "DIRECTORY",
         "name": "Output",
         "description": "Output results"
      }
   ],
   "executors": [
      {
         "image": "harbor.federated-analytics.ac.uk/5s-tes-analysis-tools/5s-tes-analysis-tools-tre-sqlpg:1.0.0",
         "command": [
            "--Output=/outputs/output.csv",
            "--Query=\nWITH last_occurrence AS (\n    SELECT\n        person_id,\n        value_as_number,\n        ROW_NUMBER() OVER (\n            PARTITION BY person_id\n            ORDER BY measurement_datetime DESC NULLS LAST\n        ) AS rn\n    FROM \"DelphiDemo\".measurement\n    WHERE measurement_concept_id = 3004249\n      AND value_as_number IS NO

'1334'

## Fetch & Collect partial statistics from each TRE output

This step fetches the approved output files from the TES submission. The `fetch_outputs()` method returns the downloaded file paths grouped by TRE.

The `collect_var_data` reads one output CSV file and extracts the row for a selected group, such as `with` or `without`. Each TRE output contains the partial statistics needed to calculate mean and variance:

- `count`: number of records in the group
- `sum`: total systolic blood pressure for the group
- `sumsq`: sum of squared systolic blood pressure values for the group

These values are stored in a `SumOfSquaresPartial` object so they can be combined across TREs.

In [ ]:
paths = wb.fetch_outputs()
data_paths = [v[0] for k, v in paths.items()]


def collect_var_data(
    path: Path,
    group_var: str = "condition_status",
    group_level: str = "with"
) -> SumOfSquaresPartial:
    data = pd.read_csv(path)

    row_data = data[data[group_var] == group_level]

    return SumOfSquaresPartial(
        count=row_data["count"].iloc[0],
        sum=row_data["sum"].iloc[0],
        sumsq=row_data["sumsq"].iloc[0],
    )

## Combine TRE outputs into a summary table

This code combines the partial statistics from each TRE for the `with` and `without` condition groups. The combined `count`, `sum`, and `sumsq` values are used to calculate the overall mean, variance, and standard deviation of systolic blood pressure.

The data returned from each TRE has information about both groups (with and without neoplasm). Our first task is to collect the data so that all the data with neoplasm is together, and all the data without neoplasm is together.


In [6]:
with_neoplasm_partials = [collect_var_data(path, group_level = "with") for path in data_paths]
without_neoplasm_partials = [collect_var_data(path, group_level = "without") for path in data_paths]

The next step is to aggregate, or combine the results from each TRE starting with the results for the group with neoplasm. The individual results can be combined with the combiner functions, which will combine the summary results and calculate the relevant statistics from them.

In [7]:
count = count_combiner.combine(with_neoplasm_partials)
mean = mean_combiner.combine(with_neoplasm_partials)
variance = variance_combiner.combine(with_neoplasm_partials)
standard_deviation = np.sqrt(variance)

with_neoplasm_data = {
    "condition_status": "with neoplasm",
    "count": count,
    "mean_systolic_blood_pressure": mean,
    "variance_systolic_blood_pressure": variance,
    "standard_deviation": standard_deviation,
    }

Similarly, for the group without neoplasm:

In [9]:
count = count_combiner.combine(without_neoplasm_partials)
mean = mean_combiner.combine(without_neoplasm_partials)
variance = variance_combiner.combine(without_neoplasm_partials)
standard_deviation = np.sqrt(variance)

without_neoplasm_data = {
    "condition_status": "without neoplasm",
    "count": count,
    "mean_systolic_blood_pressure": mean,
    "variance_systolic_blood_pressure": variance,
    "standard_deviation": standard_deviation,
    }

Now that the data has been aggregated, we can display the results using pandas.

In [10]:
summary_table = pd.DataFrame([with_neoplasm_data, without_neoplasm_data])
summary_table


,condition_status,count,mean_systolic_blood_pressure,variance_systolic_blood_pressure,standard_deviation
0,with neoplasm,1039,119.491819,3.366391,1.834773
1,without neoplasm,98375,115.692463,43.905909,6.626153
